In [0]:
%run ../connection

In [0]:
%run ./config

In [0]:
%run ./silver_utils

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DateType
 
adls_name = "adlsnewhp1"
init_silver_config(adls_name)
 
logger = get_logger("silver_trending_coins")
 
logger.info("=" * 70)
logger.info("Silver: trending_coins — START")
logger.info(f"  Run ID    : {SilverConfig.RUN_ID}")
logger.info(f"  Bronze in : {BronzePaths.TRENDING}")
logger.info(f"  Silver out: {SilverPaths.TRENDING_COINS}")
logger.info("=" * 70)
 

In [0]:

 
bronze_df, max_bronze_ts = read_bronze_incremental(
    spark, BronzePaths.TRENDING, WatermarkPaths.TRENDING,
    WatermarkPaths.WATERMARK_TABLE, logger
)
 
assert_required_columns(
    bronze_df,
    required_cols=["payload", "ingestion_timestamp", "pipeline_run_id"],
    table_name="trending_raw",
    logger=logger
)

In [0]:
display(spark.read.format("delta").load(WatermarkPaths.WATERMARK_TABLE))

In [0]:
bronze_df.display()

In [0]:

 
logger.info("CELL 3: Exploding payload.coins → one row per trending coin")
 
exploded_df = (
    bronze_df
    .select(
        F.to_date(F.col("ingestion_timestamp")).alias("trend_run_date"),
        F.col("pipeline_run_id").cast(StringType()),
        # payload.coins is an ARRAY of STRUCT {item: {id, name, ...}}
        F.explode(F.col("payload.coins")).alias("entry")
    )
)
 
logger.info(f"  Rows after explode (should be ~7): {exploded_df.count():,}")
 
 

In [0]:
exploded_df.display()

In [0]:
from pyspark.sql.types import *

In [0]:

 
logger.info("CELL 4: Extracting item fields and casting types")
 
typed_df = (
    exploded_df
    .select(
        F.col("trend_run_date").cast(DateType()),          
 
        # score is 0-based. We add 1 so position 1 = hottest trending.
        (F.col("entry.item.score").cast(IntegerType()) + 1)
         .alias("trend_position"),
 
        F.col("entry.item.id")
         .cast(StringType())
         .alias("coin_id"),                                   
 
        F.col("entry.item.name")
         .cast(StringType())
         .alias("coin_name"),
 
        F.upper(F.col("entry.item.symbol"))
         .cast(StringType())
         .alias("coin_symbol"),
 
        
        F.col("entry.item.market_cap_rank")
         .cast(IntegerType())
         .alias("market_cap_rank"),
 
        F.col("entry.item.data.price")
        .cast(DoubleType())
        .alias("coin_price"),

        F.col("entry.item.data.price_change_percentage_24h.usd")
        .cast(DoubleType())
        .alias("price_change_24h_percent"),
 
        F.col("pipeline_run_id"),
    )
    .withColumn("silver_processed_timestamp", get_silver_timestamp(SilverConfig.RUN_TS))
)
 
raw_count = typed_df.count()
logger.info(f"  Rows after extraction: {raw_count:,}")

In [0]:
typed_df.display()

In [0]:

#  DATA QUALITY FILTERS
# Trending is a short list (max 7) — if coin_id or trend_run_date is null,
# we cannot identify the coin or place it on the timeline. Drop those rows.

 
logger.info("CELL 5: Applying data quality filters")
 
clean_df = (
    typed_df
    .filter(F.col("coin_id").isNotNull())
    .filter(F.col("trend_run_date").isNotNull())
    .filter(F.col("coin_name").isNotNull())
    # trend_position must be a valid positive number
    .filter(
        F.col("trend_position").isNotNull() &
        (F.col("trend_position") >= 1)
    )
)
 
clean_count = clean_df.count()
logger.info(f"  Rows after quality filters: {clean_count:,}")
 
validate_drop_rate(
    rows_before  = raw_count,
    rows_after   = clean_count,
    max_fraction = DataQuality.MAX_DROP_FRACTION,
    table_name   = "trending_coins",
    logger       = logger,
)

In [0]:
clean_df.display()

In [0]:

 
logger.info("CELL 6: Deduplicating on trend_run_date + coin_id")
 
dedup_df    = clean_df.dropDuplicates(MergeKeys.TRENDING_COINS)
dedup_count = dedup_df.count()
logger.info(f"  Rows after dedup: {dedup_count:,} (dropped {clean_count - dedup_count:,})")

In [0]:

 
final_df = dedup_df.select(*SilverColumns.TRENDING_COINS)
log_schema(final_df, "trending_coins_final", logger)

In [0]:


 
logger.info("CELL 8: MERGE into silver/trending_coins")
 
merge_stats = delta_merge(
    spark      = spark,
    new_df     = final_df,
    table_path = SilverPaths.TRENDING_COINS,
    merge_keys = MergeKeys.TRENDING_COINS,
    logger     = logger,
)


update_watermark(
    spark          = spark,
    source_table   = WatermarkPaths.TRENDING,
    new_ts         = max_bronze_ts,
    watermark_path = WatermarkPaths.WATERMARK_TABLE,
    run_ts         = SilverConfig.RUN_TS,
    logger         = logger,
)
 

In [0]:
 

# OPTIMIZE + Z-ORDER
# Z-ORDER BY trend_run_date: Gold's trending enrichment joins by date.

 
logger.info("CELL 9: OPTIMIZE silver/trending_coins")
 
spark.sql(f"OPTIMIZE delta.`{SilverPaths.TRENDING_COINS}` ZORDER BY (trend_run_date)")
logger.info("  ✓ OPTIMIZE complete")

In [0]:

# RUN LOG + COMPLETION

 
summary = {
    "notebook"                 : "02d_silver_trending_coins",
    "pipeline_run_id"          : SilverConfig.RUN_ID,
    "run_timestamp_utc"        : SilverConfig.RUN_TS.isoformat(),
    "bronze_source"            : BronzePaths.TRENDING,
    "silver_target"            : SilverPaths.TRENDING_COINS,
    "rows_from_bronze"         : raw_count,
    "rows_after_quality_filter": clean_count,
    "rows_after_dedup"         : dedup_count,
    "merge_rows_before"        : merge_stats["rows_before"],
    "merge_rows_after"         : merge_stats["rows_after"],
    "merge_rows_inserted"      : merge_stats["rows_inserted"],
    "status"                   : "SUCCESS",
}
 
write_run_log(summary, LogPaths.TRENDING_COINS, logger)
 
logger.info("=" * 70)
logger.info("Silver: trending_coins — COMPLETE")
for k, v in summary.items():
    logger.info(f"  {k:<35}: {v}")
logger.info("=" * 70)